<a href="https://colab.research.google.com/github/seekff/learn-python/blob/main/%E8%A3%85%E9%A5%B0%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**装饰器的基础：函数是“一等公民”**

装饰器依赖 Python 函数的四个特性：

函数是对象，可赋值给变量

函数可作为参数传入另一个函数

函数内部可以定义函数（嵌套）

函数可以返回函数（闭包）

这些特性共同构成了装饰器的运行基础。



In [5]:
#函数赋值给变量
def func(message):
  print('Go a message: {}'.format(message))

send_message = func
send_message('Hello')

#函数作为参数传入另一个函数
def get_message(message):
  return 'Go a message: {}'.format(message)

def root_call(func, message):
  print(func(message))

root_call(get_message, 'hello world')

#函数里定义函数，嵌套函数
def func(message):
  def get_message(message):
    print('嵌套函数 Go a message: {}'.format(message))
  return get_message(message)

func('Good morning')

#函数的返回值是函数对象（闭包）
def func_closure():
  def get_message(message):
    print('闭包函数 Go a message: {}'.format(message))
  return get_message

send_message = func_closure()
send_message('Good afternoon')
send_message('Good evening')


Go a message: Hello
Go a message: hello world
嵌套函数 Go a message: Good morning
闭包函数 Go a message: Good afternoon
闭包函数 Go a message: Good evening



**装饰器的核心思想**

装饰器本质是一个函数，用来“包装”另一个函数，在不修改原函数代码的前提下增强其功能。

典型结构：


    def decorator(func):
      def wrapper(*args, **kwargs):
        # 执行附加逻辑
        return func(*args, **kwargs)
      return wrapper
语法糖：

    @decorator
    def func():
        ...
等价于：

    func = decorator(func)

In [7]:

def my_decorator(func):
  def wrapper():
    print('wrapper of decorator')
    func()
  return wrapper

def greet():
  print('hello world')

greet = my_decorator(greet)
greet()

@my_decorator
def greet2():
  print('hello world2')

greet2()


wrapper of decorator
hello world
wrapper of decorator
hello world2


**装饰器的多种形式**

**① 简单装饰器**

包装函数，增强行为。

**② 带参数的装饰器**

使用 *args / **kwargs 让装饰器适配任意参数。

**③ 带自定义参数的装饰器**

装饰器本身也能接收参数（典型写法是“三层函数”）：

    @repeat(4)
    def greet(...):
        ...


In [15]:
def my_decorator(func):
  def wrapper(message):
    print('wrapper of decorator')
    func(message)
  return wrapper

@my_decorator
def greet(message):
  print(message)

greet('hello world')


#使用 *args 和 **kwargs 作为装饰器内部函数wrapper()的参数。
def my_decorator2(func):
  def wrapper(*args, **kwargs):
    print('wrapper of decorator')
    func(*args, **kwargs)
  return wrapper

@my_decorator2
def greet2(message):
  print(message)

greet2('good ...')

#带有自定义参数的装饰器
def repeat(num):
  def my_decorator(func):
    def wrapper(*args, **kwargs):
      for i in range(num):
        print('wrapper of decorator')
        func(*args, **kwargs)
    return wrapper
  return my_decorator

@repeat(6)
def greet3(message):
  print(message)

greet3('hello world...装饰器带了参数6')


wrapper of decorator
hello world
wrapper of decorator
good ...
wrapper of decorator
hello world...装饰器带了参数6
wrapper of decorator
hello world...装饰器带了参数6
wrapper of decorator
hello world...装饰器带了参数6
wrapper of decorator
hello world...装饰器带了参数6
wrapper of decorator
hello world...装饰器带了参数6
wrapper of decorator
hello world...装饰器带了参数6



**保留原函数的元信息：**

使用内置的装饰器@functools.wrap



In [20]:
import functools

def my_decorator(func):
  @functools.wraps(func)
  def wrapper(*args, **kwargs):
    print('wrapper of decorator')
    func(*args, **kwargs)
  return wrapper

@my_decorator
def greet(message):
  print(message)


def my_decorator2(func):
  def wrapper(*args, **kwargs):
    print('wrapper of decorator')
    func(*args, **kwargs)
  return wrapper

@my_decorator2
def greet2(message):
  print(message)

print('保留原函数元信息:{}'.format(greet.__name__))
print('没保留原函数元信息:{}'.format(greet2.__name__))

保留原函数元信息:greet
没保留原函数元信息:wrapper


**类装饰器**

类通过实现 __call__() 也能作为装饰器使用。

典型用途：统计调用次数、状态管理等。

In [24]:
class Count:
  def __init__(self, func):
    self.func = func
    self.num_calls = 0

  def __call__(self, *args, **kwargs):
    self.num_calls += 1
    print('num of calls is: {}'.format(self.num_calls))
    return self.func(*args, **kwargs)

@Count
def example():
  print('hello world')

example()
example()
example()




num of calls is: 1
hello world
num of calls is: 2
hello world
num of calls is: 3
hello world


**装饰器的嵌套**

多个装饰器叠加时执行顺序为 从内到外


    @d1
    @d2
    @d3
    def func():
        ...
    等价于：

    d1(d2(d3(func)))

In [27]:
import functools

def my_decorator1(func):
  @functools.wraps(func)
  def wrapper(*args, **kwargs):
    print('wrapper of decorator1')
    func(*args, **kwargs)
  return wrapper

def my_decorator2(func):
  @functools.wraps(func)
  def wrapper(*args, **kwargs):
    print('wrapper of decorator2')
    func(*args, **kwargs)
  return wrapper

@my_decorator1
@my_decorator2
def greet(message):
  print(message)


greet('hello world')




wrapper of decorator1
wrapper of decorator2
hello world


**装饰器的工程应用场景**

① 身份认证（Authentication）

调用函数前检查用户是否登录。

    import functools

    def authenticate(func):
      @functools.wraps(func)
      def wrapper(*args, **kwargs):
        request = args[0]
        if check_user_logged_in(request): #日过用户处于登录状态
          return func(*args, **kwargs) #执行函数post_comment()
        else:
          raise Exception('Authentication failed')
      return wrapper

    @authenticate
    def post_comment():
      ...
      

② 日志记录（Logging）

记录函数执行时间、调用次数等。
    import functools
    import time

    def log_execution_time(func):
      @functools.wraps(func)
      def wrapper(*args, **kwargs):
        start = time.perf_counter()
        res = func(*args, **kwargs)
        end = time.perf_counter()
        print('{} took {} ms'.format(func.__name__, (end - start) * 1000))
        return res
      return wrapper

    @log_execution_time
    df callable_similarity(items):
      ...

③ 输入合理性检查（Validation）

在机器学习等场景中检查输入 JSON/参数是否合法。

④ 缓存（Caching）

使用 @lru_cache 缓存函数结果，提高性能。